### Let’s start by importing the required libraries and inspecting the structure of the news CSV files.

In [1]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import json
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer
import faiss


news_path = "C:/Users/am3xe/rag_finance_mvp/data/FinancialNewsHeadlines"

for file in os.listdir(news_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(news_path, file))
        print("\n", file)
        print(df.columns.tolist())


 cnbc_headlines.csv
['Headlines', 'Time', 'Description']

 guardian_headlines.csv
['Time', 'Headlines']

 reuters_headlines.csv
['Headlines', 'Time', 'Description']


### Extract and standardize headlines into a unified text column, then combine all news datasets.

In [2]:
news_path = "C:/Users/am3xe/rag_finance_mvp/data/FinancialNewsHeadlines"

news_dfs = []

for file in os.listdir(news_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(news_path, file))

        temp = df[["Headlines"]].rename(columns={"Headlines": "text"})
        news_dfs.append(temp)

df_news = pd.concat(news_dfs, ignore_index=True)
df_news["source"] = "Financial News"
df_news["dataset"] = "CNBC+Guardian+Reuters"


### Display random samples to verify the merged news dataset.

In [3]:
print("News samples:")
print(df_news.sample(5))

News samples:
                                                    text          source  \
31310  U.S. takes aim at judges pay in new attack on ...  Financial News   
17698  Guardian Brexit watch  How has the Brexit vote...  Financial News   
21403  Carlos Ghosn's accused smugglers unlikely to w...  Financial News   
47407  U.S. producer prices post first drop in one-an...  Financial News   
4780   Thinktank predicts £800bn hit to economy in UK...  Financial News   

                     dataset  
31310  CNBC+Guardian+Reuters  
17698  CNBC+Guardian+Reuters  
21403  CNBC+Guardian+Reuters  
47407  CNBC+Guardian+Reuters  
4780   CNBC+Guardian+Reuters  


### Load Financial PhraseBank data by separating labels from sentences.

In [4]:
fpb_path = "C:/Users/am3xe/rag_finance_mvp/data/FinancialPhraseBank-v1.0/Sentences_75Agree.txt"

fpb_texts = []

with open(fpb_path, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # split only on first whitespace occurrence
        label, sentence = line.split(maxsplit=1)
        fpb_texts.append(sentence)

df_fpb = pd.DataFrame({
    "text": fpb_texts,
    "source": "FinancialPhraseBank",
    "dataset": "FPB_75Agree"
})

print(df_fpb.head())
print("FPB rows:", len(df_fpb))

                                                text               source  \
0  to Gran , the company has no plans to move all...  FinancialPhraseBank   
1  the new production plant the company would inc...  FinancialPhraseBank   
2  the last quarter of 2010 , Componenta 's net s...  FinancialPhraseBank   
3  the third quarter of 2010 , net sales increase...  FinancialPhraseBank   
4  profit rose to EUR 13.1 mn from EUR 8.7 mn in ...  FinancialPhraseBank   

       dataset  
0  FPB_75Agree  
1  FPB_75Agree  
2  FPB_75Agree  
3  FPB_75Agree  
4  FPB_75Agree  
FPB rows: 3453


### Explore the structure of the FinQA dataset to understand its fields.

In [5]:

finqa_path = "C:/Users/am3xe/rag_finance_mvp/data/FinQA-main/dataset/train.json"

with open(finqa_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# handle list or dict
if isinstance(data, dict):
    sample = next(iter(data.values()))
else:
    sample = data[0]

print(sample.keys())
print("\nFULL SAMPLE:\n")
print(sample)


dict_keys(['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'qa', 'id', 'table_retrieved', 'text_retrieved', 'table_retrieved_all', 'text_retrieved_all'])

FULL SAMPLE:

{'pre_text': ['interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .', 'if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .', 'foreign currency exposure as more fully described in note 2i .', 'in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .', 'dollar-based exposures by entering into forward foreign currency exchange contracts .', 'the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .', 'currently , our largest foreign currency exposure is the euro , primarily because our european operations hav

### Combine pre-text and post-text from FinQA to form complete textual context.

In [6]:
finqa_path = "C:/Users/am3xe/rag_finance_mvp/data/FinQA-main/dataset"

finqa_texts = []

for split in ["train.json", "dev.json", "test.json"]:
    split_path = os.path.join(finqa_path, split)

    with open(split_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # list or dict safe
    records = data.values() if isinstance(data, dict) else data

    for item in records:
        pre = " ".join(item.get("pre_text", []))
        post = " ".join(item.get("post_text", []))

        combined_text = f"{pre} {post}".strip()

        if len(combined_text) > 100:
            finqa_texts.append(combined_text)

df_finqa = pd.DataFrame({
    "text": finqa_texts,
    "source": "FinQA",
    "dataset": "FinQA"
})

print(df_finqa.head())
print("FinQA rows:", len(df_finqa))

                                                text source dataset
0  interest rate to a variable interest rate base...  FinQA   FinQA
1  abiomed , inc . and subsidiaries notes to cons...  FinQA   FinQA
2  the following table shows annual aircraft fuel...  FinQA   FinQA
3  the fair value of our grants receivable is det...  FinQA   FinQA
4  entergy louisiana , llc management's financial...  FinQA   FinQA
FinQA rows: 8278


### Merge all datasets into one and clean by removing duplicates and missing values.

In [7]:
df_all = pd.concat(
    [df_fpb, df_news, df_finqa],
    ignore_index=True
)

df_all.drop_duplicates(subset="text", inplace=True)
df_all.dropna(inplace=True)

print("Total combined rows:", len(df_all))

Total combined rows: 59391


### Split long text into overlapping chunks and create a processed dataset for embedding.

In [8]:
def chunk_text(text, max_words=350, overlap=75):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap

    return chunks


chunked_rows = []

for _, row in tqdm(df_all.iterrows(), total=len(df_all)):
    for chunk in chunk_text(row["text"]):
        chunked_rows.append({
            "text": chunk,
            "source": row["source"],
            "dataset": row["dataset"]
        })

df_chunks = pd.DataFrame(chunked_rows)
df_chunks.drop_duplicates(subset="text", inplace=True)
df_chunks.dropna(inplace=True)

df_chunks.to_csv(
    "C:/Users/am3xe/rag_finance_mvp/data/Processed/combined_dataset_chunked.csv",
    index=False
)

print("Chunked rows:", len(df_chunks))

100%|█████████████████████████████████████████████████████████████████████████| 59391/59391 [00:02<00:00, 27273.40it/s]


Chunked rows: 64294


### Checking to ensure GPU is available for faster processing.

In [9]:
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU


### Load a pretrained sentence transformer model for generating embeddings.

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Model device:", model.device)

Model device: cuda:0


### Convert all text chunks into vector embeddings.

In [11]:
embeddings = model.encode(
    df_chunks["text"].tolist(),
    batch_size=64,          # 64–128 is usually sweet spot on consumer GPUs
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/1005 [00:00<?, ?it/s]

In [12]:
print(type(embeddings))
print(embeddings.dtype)
print(embeddings.shape)

<class 'numpy.ndarray'>
float32
(64294, 384)


### Build a FAISS index for efficient similarity search using embeddings.

In [13]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 64294


### Save the FAISS index to disk for reuse.

In [14]:
faiss.write_index(
    index,
    "C:/Users/am3xe/rag_finance_mvp/data/processed/faiss.index"
)

### Save chunked text and metadata for mapping search results.

In [15]:
df_chunks[["text", "source", "dataset"]].to_csv(
    "C:/Users/am3xe/rag_finance_mvp/data/processed/chunks_metadata.csv",
    index=False
)

### Define a function to retrieve top-k similar text chunks for a query.

In [16]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

index = faiss.read_index(
    "C:/Users/am3xe/rag_finance_mvp/data/processed/faiss.index"
)

metadata = pd.read_csv(
    "C:/Users/am3xe/rag_finance_mvp/data/processed/chunks_metadata.csv"
)

def retrieve(query, k=5):
    q_emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idx = index.search(q_emb, k)

    results = metadata.iloc[idx[0]].copy()
    results["score"] = scores[0]
    return results

### Test the retrieval system with a sample query.

In [17]:
retrieve("What happens when interest rates increase?")

,text,source,dataset,score
19587,"How will interest rate rise affect mortgages, ...",Financial News,CNBC+Guardian+Reuters,0.734740
21410,The time for the Bank to raise interest rates ...,Financial News,CNBC+Guardian+Reuters,0.648068
19563,Raising interest rates is a big mistake. This ...,Financial News,CNBC+Guardian+Reuters,0.610261
22087,Federal Reserve raises interest rates again am...,Financial News,CNBC+Guardian+Reuters,0.602000
21283,Business leader Will UK interest rates now sta...,Financial News,CNBC+Guardian+Reuters,0.596853
